# WeatherForYnov — Exploration des données climatiques

**Hackathon YNOV — Sujet 2 : Prévisions Météorologiques**

Ce notebook réalise l'acquisition, la préparation et l'analyse exploratoire des données climatiques mensuelles Météo-France (MENSQ) pour l'ensemble des départements métropolitains.

## Objectifs
- Charger et fusionner les données historiques (1950–2024) et récentes (2025–2026)
- Enrichir avec le référentiel géographique (départements → régions)
- Identifier tendances climatiques et patterns par territoire
- Exporter un jeu de données prêt pour la modélisation ML

## 1. Configuration et imports

In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100

# Chemins relatifs au dépôt (fonctionne depuis notebooks/ ou la racine)
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

DATA_DIR = ROOT / "src" / "data"
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

GEO_PATH = DATA_DIR / "referentiel_geo_departements.csv"
EXPORT_CSV = PROCESSED_DIR / "climat_mensuel_france.csv"
EXPORT_PARQUET = PROCESSED_DIR / "climat_mensuel_france.parquet"

print(f"Racine projet : {ROOT}")
print(f"Dossier données : {DATA_DIR}")

## 2. Chargement du référentiel géographique

In [ ]:
geo = pd.read_csv(GEO_PATH, sep=";")
geo["code_departement"] = geo["code_departement"].astype(str).str.zfill(2)

print(f"{len(geo)} départements dans le référentiel")
geo.groupby("nom_region")["code_departement"].count().sort_values(ascending=False)

## 3. Acquisition des données MENSQ

Les fichiers suivent la convention `MENSQ_{dept}_{periode}.csv` :
- `previous-1950-2024` : historique long
- `latest-2025-2026` : données récentes

In [ ]:
FILE_PATTERN = re.compile(r"MENSQ_(\d+)_(previous-1950-2024|latest-2025-2026)\.csv")

# Colonnes conservées pour l'analyse et le ML
RAW_COLUMNS = [
    "NUM_POSTE", "NOM_USUEL", "LAT", "LON", "ALTI", "AAAAMM",
    "RR", "QRR",           # Précipitations
    "TX", "QTX", "TXMIN",  # Température max
    "TN", "QTN", "TNMAX",  # Température min
    "TM", "QTM",           # Température moyenne journalière
    "TMM", "QTMM",         # Température moyenne mensuelle (cible ML)
    "UMM", "QUMM",         # Humidité
    "PMERM", "QPMERM",     # Pression au niveau de la mer
    "FFM", "QFFM",         # Vent moyen
    "INST", "QINST",       # Insolation
    "GLOT", "QGLOT",       # Rayonnement global
    "ETP", "QETP",         # Évapotranspiration potentielle
    "FXIAB", "QFXIAB",     # Rafale max
]


def load_mensq_files(data_dir: Path) -> pd.DataFrame:
    """Charge et concatène tous les fichiers MENSQ du dossier data."""
    frames = []
    files = sorted(data_dir.glob("MENSQ_*.csv"))

    for path in files:
        match = FILE_PATTERN.match(path.name)
        if not match:
            continue

        dept_code = match.group(1).zfill(2)
        periode = match.group(2)

        df = pd.read_csv(path, sep=";", low_memory=False)
        available = [c for c in RAW_COLUMNS if c in df.columns]
        df = df[available].copy()
        df["code_departement"] = dept_code
        df["source_periode"] = periode
        frames.append(df)

    if not frames:
        raise FileNotFoundError(f"Aucun fichier MENSQ trouvé dans {data_dir}")

    return pd.concat(frames, ignore_index=True)


raw = load_mensq_files(DATA_DIR)
print(f"Lignes chargées : {len(raw):,}")
print(f"Départements : {raw['code_departement'].nunique()}")
print(f"Stations : {raw['NUM_POSTE'].nunique()}")
raw.head(3)

## 4. Préparation et feature engineering

In [ ]:
df = raw.copy()

# Extraction année / mois depuis AAAAMM (ex: 202501 → 2025, 1)
df["annee"] = df["AAAAMM"].astype(int) // 100
df["mois"] = df["AAAAMM"].astype(int) % 100
df["date"] = pd.to_datetime(dict(year=df["annee"], month=df["mois"], day=1))

# Tendance temporelle long terme (proxy changement climatique)
df["trend_index"] = (df["annee"] - df["annee"].min()) + (df["mois"] - 1) / 12

# Features calendaires
df["trimestre"] = df["mois"].map(lambda m: (m - 1) // 3 + 1)
df["saison"] = df["mois"].map({
    12: "hiver", 1: "hiver", 2: "hiver",
    3: "printemps", 4: "printemps", 5: "printemps",
    6: "ete", 7: "ete", 8: "ete",
    9: "automne", 10: "automne", 11: "automne",
})

# Renommage des variables métier
rename_map = {
    "RR": "precipitations_mm",
    "TX": "temp_max",
    "TXMIN": "temp_max_min",
    "TN": "temp_min",
    "TNMAX": "temp_min_max",
    "TM": "temp_moy_jour",
    "TMM": "temp_moy_mensuelle",
    "UMM": "humidite",
    "PMERM": "pression_mer",
    "FFM": "vent_moyen",
    "INST": "insolation",
    "GLOT": "rayonnement_global",
    "ETP": "evapotranspiration",
    "FXIAB": "rafale_max",
}
df = df.rename(columns=rename_map)

# Conversion numérique
numeric_cols = list(rename_map.values()) + ["LAT", "LON", "ALTI"]
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# Fusion référentiel géographique
df = df.merge(geo, on="code_departement", how="left")

print(f"Période couverte : {df['annee'].min()} – {df['annee'].max()}")
df[["code_departement", "nom_departement", "nom_region", "annee", "mois", "temp_moy_mensuelle"]].head()

### 4.1 Qualité des données

Les colonnes `Q*` indiquent la qualité Météo-France : `1` = valeur valide, `0` = manquante, `9` = suspecte.

In [ ]:
quality_map = {
    "QRR": "precipitations_mm",
    "QTMM": "temp_moy_mensuelle",
    "QUMM": "humidite",
    "QFFM": "vent_moyen",
    "QPMERM": "pression_mer",
    "QINST": "insolation",
}

quality_rows = []
for qcol, vcol in quality_map.items():
    if qcol in df.columns and vcol in df.columns:
        valid = (df[qcol] == 1).sum()
        total = df[vcol].notna().sum()
        quality_rows.append({
            "variable": vcol,
            "non_null": total,
            "qualite_ok": valid,
            "taux_remplissage": round(100 * total / len(df), 1),
        })

quality_df = pd.DataFrame(quality_rows).sort_values("taux_remplissage", ascending=False)
quality_df

In [ ]:
# Jeu de travail : lignes avec température mensuelle (variable cible ML)
df_model = df[df["temp_moy_mensuelle"].notna()].copy()
print(f"Lignes exploitables (cible TMM) : {len(df_model):,} ({100*len(df_model)/len(df):.1f}%)")

# Départements présents vs référentiel
depts_data = set(df["code_departement"].unique())
depts_ref = set(geo["code_departement"].unique())
manquants = sorted(depts_ref - depts_data)
print(f"Départements sans données : {manquants or 'aucun'}")

## 5. Statistiques descriptives

In [ ]:
desc_cols = [
    "temp_moy_mensuelle", "temp_max", "temp_min",
    "precipitations_mm", "humidite", "vent_moyen",
    "pression_mer", "insolation",
]
existing_desc = [c for c in desc_cols if c in df_model.columns]
df_model[existing_desc].describe().T.round(2)

## 6. Visualisations exploratoires

### 6.1 Distribution de la température moyenne mensuelle

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df_model["temp_moy_mensuelle"], bins=40, kde=True, ax=axes[0])
axes[0].set_title("Distribution — Température moyenne mensuelle")
axes[0].set_xlabel("°C")

sns.boxplot(data=df_model, x="saison", y="temp_moy_mensuelle", order=["hiver", "printemps", "ete", "automne"], ax=axes[1])
axes[1].set_title("Température par saison")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()

### 6.2 Tendance climatique — Évolution de la température (1950–2024)

In [ ]:
# Agrégation nationale (moyenne des stations par mois)
trend_nat = (
    df_model.groupby(["annee", "mois"], as_index=False)["temp_moy_mensuelle"]
    .mean()
    .assign(date=lambda d: pd.to_datetime(dict(year=d["annee"], month=d["mois"], day=1)))
)

trend_annuelle = df_model.groupby("annee", as_index=False)["temp_moy_mensuelle"].mean()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(trend_annuelle["annee"], trend_annuelle["temp_moy_mensuelle"], marker="o", markersize=3, linewidth=1)

# Régression linéaire simple (tendance long terme)
z = np.polyfit(trend_annuelle["annee"], trend_annuelle["temp_moy_mensuelle"], 1)
p = np.poly1d(z)
ax.plot(trend_annuelle["annee"], p(trend_annuelle["annee"]), "r--", label=f"Tendance : {z[0]:.3f} °C/an")

ax.set_title("Évolution de la température moyenne annuelle — France métropolitaine")
ax.set_xlabel("Année")
ax.set_ylabel("Température moyenne (°C)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Pente de la tendance : {z[0]:.4f} °C par an")

### 6.3 Comparaison par région

In [ ]:
region_stats = (
    df_model.groupby("nom_region")["temp_moy_mensuelle"]
    .agg(["mean", "std", "count"])
    .sort_values("mean", ascending=False)
    .round(2)
)
region_stats.columns = ["temp_moy", "ecart_type", "nb_obs"]
region_stats

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
order = region_stats.index.tolist()
sns.barplot(
    data=df_model,
    y="nom_region",
    x="temp_moy_mensuelle",
    order=order,
    estimator="mean",
    errorbar="sd",
    ax=ax,
)
ax.set_title("Température moyenne mensuelle par région")
ax.set_xlabel("°C")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 6.4 Carte thermique — Température moyenne par mois et région

In [ ]:
pivot_region = (
    df_model.groupby(["nom_region", "mois"])["temp_moy_mensuelle"]
    .mean()
    .unstack("mois")
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot_region, annot=True, fmt=".1f", cmap="RdYlBu_r", center=12, ax=ax)
ax.set_title("Température moyenne (°C) — Région × Mois")
ax.set_xlabel("Mois")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

### 6.5 Précipitations et corrélations

In [ ]:
corr_cols = [c for c in [
    "temp_moy_mensuelle", "temp_max", "temp_min",
    "precipitations_mm", "humidite", "vent_moyen", "insolation",
] if c in df_model.columns]

corr = df_model[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Matrice de corrélation des variables climatiques")
plt.tight_layout()
plt.show()

### 6.6 Analyse par département — Top / Bottom températures

In [ ]:
dept_stats = (
    df_model.groupby(["code_departement", "nom_departement", "nom_region"], as_index=False)
    ["temp_moy_mensuelle"]
    .mean()
    .sort_values("temp_moy_mensuelle", ascending=False)
)

print("=== Départements les plus chauds (moyenne historique) ===")
display(dept_stats.head(10))
print("\n=== Départements les plus froids ===")
display(dept_stats.tail(10))

## 7. Agrégation régionale mensuelle

Dataset agrégé par région et mois — utile pour les visualisations interactives et le ML au niveau régional.

In [ ]:
agg_cols = {
    "temp_moy_mensuelle": "mean",
    "temp_max": "mean",
    "temp_min": "mean",
    "precipitations_mm": "mean",
    "humidite": "mean",
    "vent_moyen": "mean",
    "insolation": "mean",
}
agg_cols = {k: v for k, v in agg_cols.items() if k in df_model.columns}

df_region = (
    df_model.groupby(["code_region", "nom_region", "annee", "mois", "date", "saison", "trimestre", "trend_index"], as_index=False)
    .agg(agg_cols)
    .rename(columns={
        "temp_moy_mensuelle": "temp_moy_mensuelle_region",
        "temp_max": "temp_max_region",
        "temp_min": "temp_min_region",
        "precipitations_mm": "precipitations_mm_region",
        "humidite": "humidite_region",
        "vent_moyen": "vent_moyen_region",
        "insolation": "insolation_region",
    })
)

print(f"Lignes agrégées régionales : {len(df_region):,}")
df_region.head()

## 8. Agrégation départementale mensuelle

Une ligne par département × mois — niveau principal pour la prédiction ML.

In [ ]:
df_dept = (
    df_model.groupby(
        ["code_departement", "nom_departement", "code_region", "nom_region",
         "annee", "mois", "date", "saison", "trimestre", "trend_index"],
        as_index=False,
    )
    .agg(agg_cols)
)

print(f"Lignes agrégées départementales : {len(df_dept):,}")
df_dept.head()

## 9. Export des données préparées

In [ ]:
# Export détaillé (station × mois)
export_cols = [
    "NUM_POSTE", "NOM_USUEL", "LAT", "LON", "ALTI",
    "code_departement", "nom_departement", "code_region", "nom_region",
    "annee", "mois", "date", "saison", "trimestre", "trend_index",
    "temp_moy_mensuelle", "temp_max", "temp_min", "temp_moy_jour",
    "precipitations_mm", "humidite", "pression_mer", "vent_moyen",
    "insolation", "rayonnement_global", "evapotranspiration", "rafale_max",
    "source_periode",
]
export_cols = [c for c in export_cols if c in df_model.columns]

df_export = df_model[export_cols].copy()
df_export.to_csv(EXPORT_CSV, index=False)
df_export.to_parquet(EXPORT_PARQUET, index=False)

df_dept.to_csv(PROCESSED_DIR / "climat_mensuel_departements.csv", index=False)
df_dept.to_parquet(PROCESSED_DIR / "climat_mensuel_departements.parquet", index=False)

df_region.to_csv(PROCESSED_DIR / "climat_mensuel_regions.csv", index=False)
df_region.to_parquet(PROCESSED_DIR / "climat_mensuel_regions.parquet", index=False)

print("Fichiers exportés :")
for f in sorted(PROCESSED_DIR.glob("*")):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name} ({size_mb:.1f} Mo)")

## 10. Synthèse et prochaines étapes

### Constats principaux
- Données MENSQ couvrant **1950–2026** pour **94 départements** métropolitains
- Variable cible ML : `temp_moy_mensuelle` (TMM — température moyenne mensuelle)
- Features disponibles : min/max temp, précipitations, humidité, vent, pression, insolation + features calendaires et tendance temporelle
- Référentiel géo fusionné : départements → 13 régions administratives

### Fichiers produits (`src/data/processed/`)
| Fichier | Description |
|---------|-------------|
| `climat_mensuel_france.csv/parquet` | Données station × mois |
| `climat_mensuel_departements.csv/parquet` | Agrégation département × mois |
| `climat_mensuel_regions.csv/parquet` | Agrégation région × mois |

### Suite du projet
1. **Notebook ML** (`02_modelisation_ml.ipynb`) : prédiction mensuelle par département avec 4 semaines de données passées
2. **Application Dash/Plotly** : cartes régionales interactives avec filtres temporels
3. **Enrichissement** : données solaires (NASA), Open-Meteo pour compléter les variables manquantes